In [24]:
import numpy as np
import pandas as pd

# FASE 1

In [25]:
#bono bullet
VN = 1_000_000
T = 3
c = 0.06

VF_pasivo = 1_200_000 # a pagar en 3 años, zero-coupon, valor final
T_pasivo = T
r = 0.07

VP_pasivo = VF_pasivo / (1+r)**T #valor presente

D_mac_pasivo = T_pasivo #para zero coupon
D_mod_pasivo = D_mac_pasivo/(1+r)

print(f"Valor Presente del Pasivo: USD {VP_pasivo:,.2f}")
print(f"D. Macaulay: {D_mac_pasivo:.4f} años")
print(f"D. Modificada: {D_mod_pasivo:.4f}")

Valor Presente del Pasivo: USD 979,557.45
D. Macaulay: 3.0000 años
D. Modificada: 2.8037


# FASE 2

Utilizando 2 bonos del proyecto de Curva de Tasas, con duration modificada menor y mayor que el pasivo: AL29 (1.456) y AE38 (4.3305)

In [26]:
# Ejemplo con 2 bonos
w1, w2 = 0.40, 0.60  # pesos de mercado en la cartera
D1, D2 = 1.456, 4.3305   # duraciones modificadas de cada bono

D_cartera = w1 * D1 + w2 * D2
Gap = D_cartera - D_mod_pasivo
print(f"Duration modificada cartera antes del swap: {D_cartera:.4f}")
print(f"Duration modificada pasivo: {D_mod_pasivo:.4f}")
print(f"Gap de duración: {Gap:.4f}")
if Gap > 0:
    print("La cartera es larga, swap para acortar: pagar tasa fija / recibír flotante")
else:
    print("La cartera es corta, swap para alargar: recibir tasa fija / pagar flotante")

Duration modificada cartera antes del swap: 3.1807
Duration modificada pasivo: 2.8037
Gap de duración: 0.3770
La cartera es larga, swap para acortar: pagar tasa fija / recibír flotante


# FASE 3 Valor Swap

In [27]:
curva_spot = [0.04273, 0.04392, 0.0441] #desde derivative logic, tasas a 1, 2 y 3 años al 16/9/2026
periodos   = [1, 2, 3]
nocional   = 500_000  # provisional

In [28]:
def par_rate(curva_spot, periodos):
    descuentos = [1 / (1+curva_spot[i])**periodos[i] for i in range(len(periodos))]
    par_rate = (1-descuentos[-1])/(sum(descuentos))
    return par_rate
tasa_fija  = par_rate(curva_spot, periodos)
tasa_fija

0.04407479482189643

In [29]:
def valor_swap(nocional,tasa_fija,curva_spot,periodos):
    descuentos = [1 / (1+curva_spot[i])**periodos[i] for i in range(len(periodos))]
    flujos = [nocional*tasa_fija]*len(periodos)
    flujos[-1] += nocional
    
    B_fijo = sum(f * d for f,d in zip(flujos,descuentos))
    B_flotante = nocional
    
    V_swap = B_fijo - B_flotante
    return V_swap, B_fijo, B_flotante

V_swap, B_fijo, B_flotante = valor_swap(nocional, tasa_fija, curva_spot, periodos)
print(f"Valor swap: USD {V_swap:,.2f}")

Valor swap: USD 0.00


B_flotante = nocional solo es correcto al inicio. Si en el stress test o análisis dinámico se llama en una fecha intermedia, el bono flotante ya no vale el nocional exacto

In [30]:
def duration_swap(nocional,tasa_fija,curva_spot,periodos):
    descuentos = [1 / (1 + curva_spot[i]) ** periodos[i] for i in range(len(periodos))]
    flujos = [nocional * tasa_fija] * len(periodos)
    flujos[-1] += nocional
    
    B_fijo = sum(f * d for f, d in zip(flujos, descuentos))
    
    D_mac = sum(t*f*d for t,f,d in zip(periodos, flujos, descuentos))/B_fijo
    D_mod = D_mac / (1 + curva_spot[-1])
    
    # El bono flotante tiene D ≈ 0 (un período)
    # D_swap ≈ D_fijo (el flotante aporta mínimo)
    return D_mac, D_mod
D_mac_swap, D_mod_swap = duration_swap(nocional, tasa_fija, curva_spot, periodos)
print(f"Duration del swap: {D_mac_swap:.4f}"
      f"\nDuration modificada del swap: {D_mod_swap:.4f}")

Duration del swap: 2.8750
Duration modificada del swap: 2.7536


# FASE 4: Determinacion del Nocional

In [31]:
# Dcartera * VP_Activo + Dswap * Nswap = Dpasivo * VP_Pasivo
# Nswap = (Dpasivo * VP_Pasivo - Dcartera * VP_Activo) / Dswap
VP_activo = VP_pasivo
N_swap = (D_mod_pasivo*VP_pasivo - D_cartera * VP_activo)/D_mod_swap
print(f"Nocional del swap necesario: USD {N_swap:,.2f}")

# Verificación: duración post-swap
D_post = (D_cartera * VP_activo + D_mod_swap * N_swap) / VP_activo
print(f"Duración cartera post-swap: {D_post:.4f}")
print(f"Duración pasivo:            {D_mod_pasivo:.4f}")
print(f"Gap residual:               {D_post - D_mod_pasivo:.6f}")

Nocional del swap necesario: USD -134,099.94
Duración cartera post-swap: 2.8037
Duración pasivo:            2.8037
Gap residual:               0.000000


El nocional negativo implica un pyer swap (pagar fijo, recibir flotante), consistente con la necesidad de acortar la duration

# FASE 5: Stress Test

In [ ]:
shocks = np.arange(-0.03, 0.035, 0.005)  # -300bps a +300bps

resultados = []
for dr in shocks:
    
    curva_shock = [r + dr for r in curva_spot]
    r_shock = r + dr
    
    # Repreciar pasivo
    PV_L_shock = VF_pasivo / (1 + r_shock) ** T_pasivo
    
    # Repreciar swap con curva shockeada
    V_swap_shock, _, _ = valor_swap(N_swap, tasa_fija, curva_shock, periodos)
    # Cambio en PV del pasivo
    dL = PV_L_shock - VP_pasivo # cambio real del pasivo


    # Cambio en PV de la cartera (sin swap)
    dA_sin = -D_cartera * VP_activo * dr

    # Cambio en PV del swap
    dS = V_swap_shock - 0

    # Cambio en PV de la cartera (con swap)
    dA_con = dA_sin + dS

    resultados.append({
        'shock_bps': round(dr * 10000),
        'dL': dL,
        'dA_sin_swap': dA_sin,
        'dA_con_swap': dA_con,
        'error_sin': dA_sin - dL,
        'error_con': dA_con - dL
    })

import pandas as pd
df_stress = pd.DataFrame(resultados)
print(df_stress.to_string(index=False))

 shock_bps            dL   dA_sin_swap   dA_con_swap     error_sin     error_con
      -300  87238.178136  9.347035e+04  8.173381e+04  6.232174e+03 -5.504372e+03
      -250  71998.472597  7.789196e+04  6.820672e+04  5.893487e+03 -3.791754e+03
      -200  57047.665969  6.231357e+04  5.464031e+04  5.265902e+03 -2.407352e+03
      -150  42378.944752  4.673518e+04  4.103554e+04  4.356231e+03 -1.343409e+03
      -100  27985.687370  3.115678e+04  2.739330e+04  3.171097e+03 -5.923860e+02
       -50  13861.457895  1.557839e+04  1.371450e+04  1.716934e+03 -1.469569e+02
         0      0.000000 -2.161936e-11 -2.161936e-11 -2.161936e-11 -2.161936e-11
        50 -13604.768857 -1.557839e+04 -1.374936e+04 -1.973623e+03 -1.445923e+02
       100 -26958.763045 -3.115678e+04 -2.753277e+04 -4.198021e+03 -5.740024e+02
       150 -40067.734152 -4.673518e+04 -4.134942e+04 -6.667442e+03 -1.281685e+03
       200 -52937.276196 -6.231357e+04 -5.519855e+04 -9.376292e+03 -2.261274e+03
       250 -65572.830632 -7.

Se asume un shock paralelo uniforme sobre ambas curvas (r=0.07 y SOFR)

# FASE 6: Excel

In [33]:
df_stress.to_csv("df_stress.csv", index=False) #resto de vlores hechos manualmente